# RASTA CAD OCTA Classification — Phase 1 Dataset Analysis

This notebook is the runnable project entry point. Phase 1 audits the RASTA OCTA dataset and clinical metadata before any modelling.

**Important:** patient-level splitting and modelling are intentionally not executed in this phase. Set `DATA_ROOT` to the RASTA dataset folder on the GPU machine before running.

In [ ]:
from pathlib import Path
import os
import sys
import json
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from rasta_cad.data import (
    scan_image_manifest, load_clinical_table, attach_clinical,
    default_clinical_columns, dataset_audit
)
from rasta_cad.splits import assign_patient_folds, validate_patient_folds
from rasta_cad.utils import seed_everything, ensure_dir, save_json

seed_everything(42)
sns.set_theme(style='whitegrid', context='notebook')

## Configuration

Set `RASTA_DATA_ROOT` as an environment variable or edit `DATA_ROOT` below. The local development machine does not contain the image dataset, so this cell is designed for the GPU/data machine.

In [ ]:
DATA_ROOT = Path(os.environ.get('RASTA_DATA_ROOT', 'Rasta dataset'))
CLINICAL_PATH = Path(os.environ.get('RASTA_CLINICAL_PATH', 'table_for_publication .xlsx'))
OUTPUT_DIR = ensure_dir(PROJECT_ROOT / 'outputs' / 'phase1_dataset_analysis')
REQUIRE_ALL_LAYERS_FOR_MODEL = True
QC_MAX_IMAGES_PER_LAYER = None  # set e.g. 500 for a quicker smoke-test audit

print('DATA_ROOT:', DATA_ROOT.resolve() if DATA_ROOT.exists() else DATA_ROOT)
print('CLINICAL_PATH:', CLINICAL_PATH.resolve() if CLINICAL_PATH.exists() else CLINICAL_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR.resolve())

## Load clinical metadata

The provided publication spreadsheet contains demographics, CAD risk factors, and precomputed OCTA biomarkers. Missingness is preserved for later learnable mask-token handling.

In [ ]:
clinical_df = load_clinical_table(CLINICAL_PATH)
print('Clinical shape:', clinical_df.shape)
display(clinical_df.head())
print('Columns:')
print(list(clinical_df.columns))

## Scan OCTA files and create patient-eye manifest

Each row is one eye. OD/OS are represented separately but retain the same `patient_id` for leakage-safe fold assignment later. The scanner tolerates RASTA naming variants such as `cc OD.bmp`, `Cc_OD.bmp`, and case differences.

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f'DATA_ROOT does not exist: {DATA_ROOT}. '
        'Set the RASTA_DATA_ROOT environment variable or edit DATA_ROOT in the configuration cell.'
    )

manifest_all = scan_image_manifest(DATA_ROOT, require_all_layers=False)
manifest = scan_image_manifest(DATA_ROOT, require_all_layers=REQUIRE_ALL_LAYERS_FOR_MODEL)
manifest_clin = attach_clinical(manifest, clinical_df)

manifest_all.to_csv(OUTPUT_DIR / 'manifest_all_detected_eyes.csv', index=False)
manifest_clin.to_csv(OUTPUT_DIR / 'manifest_model_ready_with_clinical.csv', index=False)

print('All detected eye rows:', manifest_all.shape)
print('Model-ready rows with all layers:', manifest.shape)
display(manifest_clin.head())

## Core dataset audit

This covers total samples, cohort imbalance, bilateral availability, layer availability, clinical missingness, and possible patient leakage across cohorts.

In [ ]:
clinical_cols = default_clinical_columns(manifest_clin)
audit = dataset_audit(manifest_clin, clinical_cols=clinical_cols)
save_json(audit, OUTPUT_DIR / 'dataset_audit.json')

print(json.dumps(audit, indent=2, default=str))

class_eye = manifest_clin['cohort'].value_counts().rename_axis('cohort').reset_index(name='eye_count')
class_patient = manifest_clin.drop_duplicates('patient_id')['cohort'].value_counts().rename_axis('cohort').reset_index(name='patient_count')
class_table = class_patient.merge(class_eye, on='cohort', how='outer')
class_table.to_csv(OUTPUT_DIR / 'class_distribution.csv', index=False)
display(class_table)

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=class_table, x='cohort', y='patient_count', color='#4C72B0')
plt.xticks(rotation=30, ha='right')
plt.title('Patient count per cohort')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution_patients.png', dpi=200)
plt.show()

eyes_per_patient = manifest_clin.groupby('patient_id')['eye'].agg(lambda s: '+'.join(sorted(set(s))))
bilat = eyes_per_patient.value_counts().rename_axis('eye_availability').reset_index(name='patient_count')
bilat.to_csv(OUTPUT_DIR / 'bilateral_availability.csv', index=False)
display(bilat)

## Clinical missingness and demographic distributions

In [ ]:
missing_overall = manifest_clin[clinical_cols].isna().mean().sort_values(ascending=False).rename('missing_fraction')
missing_by_cohort = manifest_clin.groupby('cohort')[clinical_cols].apply(lambda g: g.isna().mean()).reset_index()
missing_overall.to_csv(OUTPUT_DIR / 'clinical_missing_overall.csv')
missing_by_cohort.to_csv(OUTPUT_DIR / 'clinical_missing_by_cohort.csv', index=False)
display(missing_overall.to_frame())
display(missing_by_cohort)

plt.figure(figsize=(12, max(4, 0.025 * len(manifest_clin))))
sns.heatmap(manifest_clin[clinical_cols].isna(), cbar=False, yticklabels=False)
plt.title('Missing clinical data heatmap (eye rows × features)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'clinical_missingness_heatmap.png', dpi=200)
plt.show()

In [ ]:
demo_cols = [c for c in ['Age', 'Sex', 'Body mass index'] if c in manifest_clin.columns]
demo_summary = manifest_clin.drop_duplicates('patient_id').groupby('cohort')[demo_cols].agg(['count', 'mean', 'std', 'median', 'min', 'max'])
demo_summary.to_csv(OUTPUT_DIR / 'demographic_summary_by_cohort.csv')
display(demo_summary)

if 'Age' in manifest_clin.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=manifest_clin.drop_duplicates('patient_id'), x='cohort', y='Age')
    plt.xticks(rotation=30, ha='right')
    plt.title('Age distribution per cohort')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'age_distribution_by_cohort.png', dpi=200)
    plt.show()

if 'Body mass index' in manifest_clin.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=manifest_clin.drop_duplicates('patient_id'), x='cohort', y='Body mass index')
    plt.xticks(rotation=30, ha='right')
    plt.title('BMI distribution per cohort')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'bmi_distribution_by_cohort.png', dpi=200)
    plt.show()

## Image resolution and lightweight quality audit

This reads OCTA images to summarize resolution, intensity range, standard deviation, and zero-variance/unreadable counts. It does **not** train or run any model.

In [ ]:
def audit_image_paths(df, max_per_layer=None):
    rows = []
    for layer, col in [('sup', 'sup_path'), ('deep', 'deep_path'), ('cc', 'cc_path')]:
        paths = df[col].dropna().tolist()
        if max_per_layer is not None:
            paths = paths[:max_per_layer]
        for p in paths:
            rec = {'layer': layer, 'path': p}
            try:
                with Image.open(p) as img:
                    arr = np.asarray(img.convert('L'), dtype=np.float32)
                rec.update({
                    'height': int(arr.shape[0]), 'width': int(arr.shape[1]),
                    'mean': float(arr.mean()), 'std': float(arr.std()),
                    'min': float(arr.min()), 'max': float(arr.max()),
                    'zero_variance': bool(arr.std() <= 1e-6),
                    'readable': True,
                })
            except Exception as e:
                rec.update({'readable': False, 'error': repr(e)})
            rows.append(rec)
    return pd.DataFrame(rows)

image_qc = audit_image_paths(manifest_clin, max_per_layer=QC_MAX_IMAGES_PER_LAYER)
image_qc.to_csv(OUTPUT_DIR / 'image_quality_audit.csv', index=False)
display(image_qc.head())

qc_summary = image_qc.groupby('layer').agg(
    n=('path', 'count'),
    readable=('readable', 'sum'),
    zero_variance=('zero_variance', 'sum'),
    median_height=('height', 'median'),
    median_width=('width', 'median'),
    median_std=('std', 'median'),
).reset_index()
qc_summary.to_csv(OUTPUT_DIR / 'image_quality_summary.csv', index=False)
display(qc_summary)

if not image_qc.empty and 'std' in image_qc:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=image_qc[image_qc['readable']], x='layer', y='std')
    plt.title('Image intensity standard deviation by OCTA layer')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'image_std_by_layer.png', dpi=200)
    plt.show()

## Biomarker distributions

The clinical spreadsheet includes OCTA biomarker columns such as density, perfusion, and FAZ metrics. These are summarized and plotted by cohort as available.

In [ ]:
biomarker_pattern = re.compile(r'^(FAZ|Faz|DENS_|PERF_)')
biomarker_cols = [c for c in manifest_clin.columns if biomarker_pattern.search(str(c))]
print('Detected biomarker columns:', len(biomarker_cols))
print(biomarker_cols[:20])

if biomarker_cols:
    biomarker_summary = manifest_clin.groupby('cohort')[biomarker_cols].agg(['count', 'mean', 'std', 'median'])
    biomarker_summary.to_csv(OUTPUT_DIR / 'biomarker_summary_by_cohort.csv')
    display(biomarker_summary.iloc[:, :min(16, biomarker_summary.shape[1])])

    plot_cols = biomarker_cols[:12]
    for col in plot_cols:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=manifest_clin, x='cohort', y=col)
        plt.xticks(rotation=30, ha='right')
        plt.title(f'{col} by cohort')
        plt.tight_layout()
        safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', col)
        plt.savefig(OUTPUT_DIR / f'biomarker_boxplot_{safe}.png', dpi=200)
        plt.show()

## Confounder and leakage checks

We inspect age/sex imbalance and verify that no patient ID appears in multiple cohorts. A preliminary patient-level stratified fold assignment is generated for review only.

In [ ]:
patient_level = manifest_clin.drop_duplicates('patient_id').copy()
leakage = patient_level.groupby('patient_id')['cohort'].nunique()
leaked_patients = leakage[leakage > 1].index.tolist()
print('Patients appearing in multiple cohorts:', len(leaked_patients))
print(leaked_patients[:20])

confounder_tables = {}
if 'Age' in patient_level.columns:
    confounder_tables['age_by_cohort'] = patient_level.groupby('cohort')['Age'].describe()
    display(confounder_tables['age_by_cohort'])
if 'Sex' in patient_level.columns:
    confounder_tables['sex_by_cohort'] = pd.crosstab(patient_level['cohort'], patient_level['Sex'], normalize='index')
    display(confounder_tables['sex_by_cohort'])

for name, table in confounder_tables.items():
    table.to_csv(OUTPUT_DIR / f'{name}.csv')

fold_df = assign_patient_folds(manifest_clin, n_splits=5, seed=42, age_col='Age' if 'Age' in manifest_clin else None, sex_col='Sex' if 'Sex' in manifest_clin else None)
validate_patient_folds(fold_df, n_splits=5)
fold_df[['patient_id', 'eye', 'cohort', 'label', 'fold']].to_csv(OUTPUT_DIR / 'preliminary_patient_level_folds.csv', index=False)
display(pd.crosstab(fold_df['fold'], fold_df['cohort']))
print('Zero patient overlap across folds verified.')

## Phase 1 completion checklist

After running this notebook section on the dataset machine, review the files in `outputs/phase1_dataset_analysis/` before proceeding to Phase 2 preprocessing and dataset object finalization.